In [ ]:
import numpy as np,matplotlib.pyplot as plt,tensorflow as tf
np.random.seed(15);tf.random.set_seed(15)
# Three years of synthetic daily closing prices as a reproducible stock-index proxy
n=3*365;returns=np.random.normal(.0004,.012,n);prices=1000*np.exp(np.cumsum(returns))
def gradient_by_time(length,activation):
 x=tf.constant(prices[-length:].reshape(1,length,1),dtype=tf.float32)
 layer=tf.keras.layers.SimpleRNN(16,activation=activation,return_sequences=True)
 _=layer(x)
 with tf.GradientTape() as tape:
  states=layer(x);tape.watch(states);loss=tf.reduce_mean(tf.square(states[:,-1,:]))
 grad=tape.gradient(loss,states)
 return tf.norm(grad,axis=-1).numpy().ravel()
fig,ax=plt.subplots(1,3,figsize=(14,4))
for j,L in enumerate([10,50,100]):
 for act in ["tanh","relu"]:
  g=gradient_by_time(L,act);ax[j].plot(np.arange(1,L+1),g,label=act)
 ax[j].set_yscale("log");ax[j].set_title(f"Sequence length {L} days");ax[j].set_xlabel("Time step");ax[j].set_ylabel("Gradient norm");ax[j].legend();ax[j].grid()
plt.tight_layout();plt.show()
for L in [10,50,100]:
 for act in ["tanh","relu"]:
  g=gradient_by_time(L,act);print(f"Length={L}, activation={act}, first gradient={g[0]:.8g}, last gradient={g[-1]:.8g}")
plt.figure(figsize=(10,3));plt.plot(prices);plt.title("Three-year simulated daily stock-index closing prices");plt.xlabel("Day");plt.ylabel("Index");plt.grid();plt.show()